### Define a function for making per-cell data into per-cell with neighbors data.
This makes it so each cell has the information of each of it's 8 neighbors in it's row too.

In [1]:
import json

import geopandas as gpd
import gradio as gr
import h3
import numpy as np
import pandas as pd
import plotly.express as px
from shapely.geometry import shape
from pathlib import Path
import re

def build_3x3_neighborhood_df(
    df: pd.DataFrame,
    band_cols: list[str],
    cell_size: float = 20.0,
    group_cols: list[str] | None = None,
) -> pd.DataFrame:
    """
    Returns one row per center cell with all original columns plus neighbor-band features.

    - Keeps every original column from df unchanged.
    - Adds only the 8 surrounding neighbor band sets as new columns.
    - Center-cell bands remain original names (B11, B12, ...), no duplicated p0_p0 columns.
    - Uses inner joins for neighbors, so edge/missing-neighbor cells are dropped.
    """
    if group_cols is None:
        group_cols = [c for c in ["year"] if c in df.columns]

    required_cols = set(["x", "y", *band_cols, *group_cols])
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    merge_keys = [*group_cols, "x", "y"]

    # Keep all original columns as the base output
    out = df.copy()

    def offset_suffix(dx: int, dy: int) -> str:
        sx = f"{'m' if dx < 0 else 'p'}{abs(dx)}"
        sy = f"{'m' if dy < 0 else 'p'}{abs(dy)}"
        return f"{sx}_{sy}"

    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            # Skip center: those band columns already exist in out
            if dx == 0 and dy == 0:
                continue

            neigh = df[[*group_cols, "x", "y", *band_cols]].copy()

            # Shift neighbor coordinates into center coordinate frame
            neigh["x"] = neigh["x"] - dx * cell_size
            neigh["y"] = neigh["y"] - dy * cell_size

            suff = offset_suffix(dx, dy)
            neigh = neigh.rename(columns={c: f"{c}_{suff}" for c in band_cols})

            out = out.merge(neigh, on=merge_keys, how="inner")

    return out



### Actually use the function to convert all the data into the 3x3 format.

In [ ]:
data_path = "data"
original_files = list(Path(data_path).iterdir())
output_files = [Path(str(og.parent.parent.absolute() / "data_3x3" / og.stem) + "_3x3" + str(og.suffix)) for og in original_files]

for in_file, out_file in zip(original_files, output_files):

    # Skip if the output file already exists
    if out_file.exists() and out_file.is_file():
        print(f"{out_file} already exists. Skipping...")
        continue

    print(f"Starting on input: {in_file.name}; output: {out_file.name}")
    # Load the CSV
    df = pd.read_csv(in_file, index_col=False)
    df.drop(columns=["class_label"], inplace=True)

    # Parse the GeoJSON
    df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

    # Example: infer band and output columns from your current schema
    band_columns = [c for c in df.columns if re.fullmatch(r"B\d+", c)]

    df_3x3 = build_3x3_neighborhood_df(
        df=df,
        band_cols=band_columns,
        cell_size=20.0,
        group_cols=["year"] if "year" in df.columns else None,
    )

    print("Original rows:", len(df))
    print("3x3 valid center rows:", len(df_3x3))
    print("Row diff:", len(df) - len(df_3x3))
    print("Feature columns:", len(df_3x3.columns))
    print(*df.columns)
    print(*df_3x3.columns)

    print(f"Saving {out_file}...")
    df_3x3.to_csv(out_file, index=False)

/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2016_mit_2021_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2018_mit_2020_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2019_mit_2020_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2017_mit_2021_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2016_mit_2020_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2018_mit_2021_labels_3x3.csv already exists. Skipping...
/home/macedmo/git/machine-learning-final-project-ws2025/dashboard/data_3x3/df_2017_mit_2020_labels_3x3.csv already exists. Skipping...
Starting on input: df_2020_mit_2021_labels.csv; output:

#### Actually use the function to convert all the data into the 3x3 format.

In [15]:
data_path = "data"
original_files = [f for f in Path(data_path).iterdir() if f.is_file() and f.suffix == ".csv"]
output_dir = Path("data_3x3_stress_test")
output_dir.mkdir(parents=True, exist_ok=True)

def extract_years(filename: Path):
    parts = filename.stem.split("_")
    return int(parts[1]), int(parts[3])

for sigma_scale in [0.05, 0.1, 0.25]:
    rng = np.random.default_rng(42)
    grouped_by_label_year = {}

    for in_file in original_files:
        print(f"Processing input: {in_file.name} (sigma={sigma_scale:.2f})")

        # Load the CSV
        df = pd.read_csv(in_file, index_col=False)
        df.drop(columns=["class_label"], inplace=True, errors="ignore")

        # Parse the GeoJSON
        df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

        # Add noise to Sentinel bands
        band_columns = [c for c in df.columns if re.fullmatch(r"B\d+", c)]
        noise = rng.normal(
            loc=0.0,
            scale=(df[band_columns].std(axis=0, ddof=0).replace(0, 1.0) * sigma_scale).values,
            size=(len(df), len(band_columns)),
        )

        df_noisy = df.copy()
        df_noisy.loc[:, band_columns] = df_noisy[band_columns].to_numpy() + noise

        df_3x3 = build_3x3_neighborhood_df(
            df=df_noisy,
            band_cols=band_columns,
            cell_size=20.0,
            group_cols=["year"] if "year" in df.columns else None,
        )

        start_year, label_year = extract_years(in_file)
        df_3x3.insert(0, "delta_years", label_year - start_year)

        grouped_by_label_year.setdefault(label_year, []).append(df_3x3)
        print("3x3 valid center rows:", len(df_3x3))

    for label_year, dfs in grouped_by_label_year.items():
        dfo = pd.concat(dfs, ignore_index=True)
        out_file = output_dir / f"delta_table_{label_year}_3x3_{int(sigma_scale * 100):d}_percent.csv"
        print(f"Saving {out_file} with shape {dfo.shape}...")
        dfo.to_csv(out_file, index=False)

Processing input: df_2017_mit_2020_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2016_mit_2020_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2018_mit_2021_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2020_mit_2021_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2017_mit_2021_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2019_mit_2021_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2021_mit_2021_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2018_mit_2020_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2019_mit_2020_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2020_mit_2020_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Processing input: df_2016_mit_2021_labels.csv (sigma=0.05)
3x3 valid center rows: 457363
Saving data_3x3_stres

In [ ]:
df = pd.read_csv("data/df_2016_mit_2020_labels.csv", index_col=False)
df.drop(columns=["class_label"], inplace=True)

# Columns requested for Gaussian noise
noise_columns = [
    "B11", "B12", "B2", "B3", "B4", "B5", "B6", "B7", "B8",
]

# Reproducible Gaussian noise: 5% of each column's std dev
rng = np.random.default_rng(42)
for sigma_scale in [0.05, 0.1, 0.25]:
    noise = rng.normal(
        loc=0.0,
        scale=(df[noise_columns].std(axis=0, ddof=0).replace(0, 1.0) * sigma_scale).values,
        size=(len(df), len(noise_columns)),
    )

    df_noisy = df.copy()
    df_noisy.loc[:, noise_columns] = df_noisy[noise_columns].to_numpy() + noise

    print("Noisy columns:", noise_columns)
    print(df[noise_columns].head(2))
    print(df_noisy[noise_columns].head(2))

Noisy columns: ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8']
      B11    B12     B2     B3     B4     B5      B6      B7      B8
0  1371.5  608.5  114.5  262.5  161.5  582.0  1772.0  2434.5  2296.5
1  1323.0  577.5  102.5  228.5  152.0  503.0  1890.0  2166.5  2188.5
           B11         B12          B2          B3          B4          B5  \
0  1380.329185  577.770660  130.322107  283.020180  111.262229  549.527816   
1  1298.283031  603.484358  118.898544  229.940583  181.025661  514.658185   

            B6           B7           B8  
0  1776.117124  2422.694804  2295.842891  
1  1862.326323  2180.265303  2150.997184  
Noisy columns: ['B11', 'B12', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8']
      B11    B12     B2     B3     B4     B5      B6      B7      B8
0  1371.5  608.5  114.5  262.5  161.5  582.0  1772.0  2434.5  2296.5
1  1323.0  577.5  102.5  228.5  152.0  503.0  1890.0  2166.5  2188.5
           B11         B12          B2          B3          B4          B5  \


### Combine all of the data from all the tables for a given label year, into large tables with "delta_year" offsets from the label year.

In [27]:
data_path = "data_3x3"
label_year = 2021
files = list(Path(data_path).iterdir())

filtered_files = [file for file in files if f"mit_{label_year}_labels" in str(file)]

def extract_years(filename):
    f = str(filename.stem).split("_")
    return int(f[1]), int(f[3])

dfs = []

for file in filtered_files:
    df = pd.read_csv(file, index_col=False)
    start_year, end_year = extract_years(file)
    df.insert(0, "delta_years", end_year-start_year)
    dfs.append(df)

dfo = pd.concat(dfs)

for df in dfs:
    print(df.shape)

print(dfo.shape)

dfo.to_csv(filtered_files[0].parent / f"delta_table_{label_year}_3x3.csv", index=False)

(457363, 96)
(457363, 96)
(457363, 96)
(457363, 96)
(457363, 96)
(457363, 96)
(2744178, 96)


In [28]:
dfo["delta_years"].unique()

array([2, 3, 1, 0, 5, 4])

In [30]:
dfo.sample(5)

,delta_years,system:index,B11,B12,B2,B3,B4,B5,B6,B7,...,B8_p0_p1,B11_p1_p1,B12_p1_p1,B2_p1_p1,B3_p1_p1,B4_p1_p1,B5_p1_p1,B6_p1_p1,B7_p1_p1,B8_p1_p1
162987,4,167259,1827.5,1622.5,501.0,670.0,746.5,985.0,1533.0,1663.0,...,1655.0,1856.0,1670.0,497.0,650.0,759.0,1059.5,1375.5,1291.0,1234.0
411670,4,419603,2105.0,1870.0,1723.0,1770.0,1748.0,1809.0,1742.0,1761.0,...,1232.0,1899.0,1615.0,828.0,1009.0,1080.0,1399.0,1622.0,1845.0,1555.0
257643,2,263749,1988.5,1005.5,260.0,439.5,351.0,847.0,2531.0,3123.0,...,2519.5,1582.0,1013.0,385.5,492.0,432.5,678.0,1502.5,1875.5,1808.5
414271,5,422279,2113.0,1748.0,332.0,557.0,586.0,1120.0,2160.0,2301.0,...,2562.0,2033.0,1143.0,301.0,713.0,394.0,1181.0,3791.0,4398.0,4256.0
441391,4,449775,2325.0,2037.0,627.0,849.0,1129.0,1369.0,2771.0,3200.0,...,3315.0,2430.0,2068.0,666.0,895.0,1222.0,1425.0,2815.0,3214.0,3213.0


### Experiment with gathering data from Open Street Maps.

In [4]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
from shapely import wkt

import json
from shapely.geometry import shape

# Updated parsing function
def parse_geometry(geo_str):
    if pd.isna(geo_str) or not isinstance(geo_str, str):
        return None
    try:
        # 1. Parse the string into a Python dictionary
        geo_dict = json.loads(geo_str)
        # 2. Convert that dictionary into a Shapely object
        return shape(geo_dict)
    except:
        return None

# 1. Load your data
df = pd.read_csv("data.csv") # Assuming your sample is in a CSV
df['geometry'] = df['.geo'].apply(parse_geometry)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:32632")

# 2. Define your target area (Nuremberg) for OSM data
place_name = "Nuremberg, Germany"

# 3. Fetch Road and Water data from OpenStreetMap
# Fetching "primary" and "secondary" roads as they are major change drivers
roads = ox.features_from_place(place_name, tags={"highway": ["primary", "secondary", "tertiary"]})
water = ox.features_from_place(place_name, tags={"natural": "water", "waterway": True})

# 4. Project OSM data to match your UTM 32N (EPSG:32632) for meter-based distance
roads = roads.to_crs("EPSG:32632")
water = water.to_crs("EPSG:32632")

# 5. Create a "Built-up" reference from your OWN 2016 data
# This is crucial for predicting change based on existing urban footprints [cite: 104]
existing_urban = gdf[gdf['built_up'] > 0.5].unary_union

def get_min_dist(point, target_gdf):
    """Calculates minimum distance from a point to any object in a GeoDataFrame."""
    return target_gdf.distance(point).min()

# 6. Calculate proximity features
print("Calculating distances... this may take a moment.")
gdf['dist_to_road'] = gdf.geometry.centroid.apply(lambda x: get_min_dist(x, roads))
gdf['dist_to_water'] = gdf.geometry.centroid.apply(lambda x: get_min_dist(x, water))
gdf['dist_to_builtup'] = gdf.geometry.centroid.apply(lambda x: x.distance(existing_urban))

print(gdf[['dist_to_road', 'dist_to_water', 'dist_to_builtup']].head())

/tmp/ipykernel_293076/3264745948.py:41: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  existing_urban = gdf[gdf['built_up'] > 0.5].unary_union


Calculating distances... this may take a moment.
   dist_to_road  dist_to_water  dist_to_builtup
0    349.033473      74.740731       496.588361
1    360.386728      93.039641       505.371151
2    372.468552     111.922174       514.781507
3    385.210399     131.136460       524.785671
4    398.548968     150.555535       535.350353


In [5]:
gdf.to_csv("data_with_extras.csv")

### Experiment with loading ML Models

In [5]:
import json
import pickle
from pathlib import Path

import geopandas as gpd
import gradio as gr
import numpy as np
import pandas as pd
import plotly.express as px
from shapely.geometry import box, shape

def load_prediction_model(model_path="artifacts/xgboost_multioutput.pkl"):
    model_file = Path(model_path)
    if not model_file.exists():
        return None

    with model_file.open("rb") as f:
        return pickle.load(f)

model = load_prediction_model()

In [4]:
model.estimators_

[XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.678024823943575, device='cuda',
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=0.4287949729173641, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.033161675945190996,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=8, max_leaves=None,
              min_child_weight=8, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=226, n_jobs=-1,
              num_parallel_tree=None, ...),
 XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.678024823943575, device='cuda',
 

In [6]:

def load_data_from_csv(csv_path="data.csv", cell_size_m=200):
    # 1. Load CSV data
    df = pd.read_csv(csv_path)

    # 2. Parse geometry from GeoJSON strings
    df["geometry"] = df[".geo"].apply(lambda x: shape(json.loads(x)))

    # 3. Build GeoDataFrame in projected CRS (meters)
    gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:32632")

    # Aggregate 20x20m cells into larger display cells.
    gdf["grid_x"] = (np.floor(gdf["x"] / cell_size_m) * cell_size_m).astype(int)
    gdf["grid_y"] = (np.floor(gdf["y"] / cell_size_m) * cell_size_m).astype(int)

    # agg_df = (
    #     gdf.groupby(["grid_x", "grid_y"], as_index=False)
    #     .agg({**{column: "mean" for column in class_cols}, "cell_id": "count"})
    #     .rename(columns={"cell_id": "cell_count"})
    # )

    numeric_columns = gdf.select_dtypes(include=[np.number]).columns.tolist()
    agg_columns = {
        column: "mean"
        for column in numeric_columns
        if column not in {"grid_x", "grid_y"}
    }
    agg_df = gdf.groupby(["delta_years", "grid_x", "grid_y"], as_index=False).agg(
        agg_columns
    )

    agg_df["geometry"] = agg_df.apply(
        lambda row: box(
            row["grid_x"],
            row["grid_y"],
            row["grid_x"] + cell_size_m,
            row["grid_y"] + cell_size_m,
        ),
        axis=1,
    )

    gdf = gpd.GeoDataFrame(agg_df, geometry="geometry", crs="EPSG:32632")

    # 4. Reproject to latitude/longitude
    gdf = gdf.to_crs("EPSG:4326")

    # 5. Create unique IDs for Plotly linking
    gdf["Hexagon_ID"] = gdf.index

    # Add vegetation target label
    gdf["vegetation"] = gdf[["tree_cover", "cropland", "grassland"]].sum(axis=1)

    # Add engineered features
    gdf["NDVI"] = (gdf["B8"] - gdf["B4"]) / (gdf["B8"] + gdf["B4"] + 1e-8)
    gdf["EVI2"] = 2.5 * (
        (gdf["B8"] - gdf["B4"]) / (gdf["B8"] + 2.4 * gdf["B4"] + 1 + 1e-8)
    )
    gdf["SAVI"] = ((gdf["B8"] - gdf["B4"]) * 1.5) / (gdf["B8"] + gdf["B4"] + 0.5 + 1e-8)
    gdf["NDBI"] = (gdf["B11"] - gdf["B8"]) / (gdf["B11"] + gdf["B8"] + 1e-8)
    gdf["NDWI"] = (gdf["B3"] - gdf["B8"]) / (gdf["B3"] + gdf["B8"] + 1e-8)
    gdf["MNDWI"] = (gdf["B3"] - gdf["B11"]) / (gdf["B3"] + gdf["B11"] + 1e-8)

    # 6. Convert to GeoJSON format expected by Plotly
    plotly_geojson = json.loads(gdf.geometry.to_json())
    return gdf, plotly_geojson

In [10]:
gdf, _ = load_data_from_csv("data_3x3/delta_table_2021_3x3.csv")

In [13]:
base_df = gdf[gdf["delta_years"].astype(int) == 1].copy()

In [14]:
base_df

,grid_x,grid_y,delta_years,system:index,B11,B12,B2,B3,B4,B5,...,B8_p1_p1,geometry,Hexagon_ID,vegetation,NDVI,EVI2,SAVI,NDBI,NDWI,MNDWI
4947,643800,5488800,1.0,27073.500000,1723.000000,1291.000000,475.000000,728.000000,733.000000,1114.000000,...,2590.000000,"POLYGON ((10.99021 49.53468, 10.99028 49.53648...",4947,0.985368,0.425998,0.759522,0.638873,-0.027652,-0.428796,-0.405957
4948,643800,5489000,1.0,27869.266667,2563.866667,2077.066667,623.200000,927.266667,1162.866667,1494.400000,...,2731.866667,"POLYGON ((10.99028 49.53648, 10.99036 49.53828...",4948,0.913529,0.343087,0.587430,0.514558,0.037709,-0.438836,-0.468788
4949,643800,5489200,1.0,28675.200000,2795.666667,2326.333333,678.400000,975.666667,1261.733333,1632.966667,...,2545.500000,"POLYGON ((10.99036 49.53828, 10.99043 49.54008...",4949,1.000000,0.346184,0.593623,0.519209,0.036674,-0.453948,-0.482588
4950,643800,5489400,1.0,29020.400000,2502.000000,1790.000000,577.200000,888.400000,1008.600000,1628.400000,...,3425.800000,"POLYGON ((10.99043 49.54008, 10.9905 49.54188,...",4950,1.000000,0.534585,1.007875,0.801786,-0.141327,-0.578358,-0.475932
4951,644000,5488800,1.0,27021.235294,2056.882353,1240.647059,448.705882,764.352941,647.941176,1269.588235,...,3392.411765,"POLYGON ((10.99297 49.53464, 10.99305 49.53644...",4951,0.880500,0.694121,1.428996,1.041059,-0.271323,-0.648816,-0.458143
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9889,665200,5476200,1.0,268.821429,1123.517857,580.196429,260.258929,379.125000,282.383929,616.964286,...,2236.696429,"POLYGON ((11.28048 49.41598, 11.28056 49.41777...",9889,1.000000,0.766476,1.646380,1.149476,-0.310638,-0.698532,-0.495389
9890,665200,5476400,1.0,668.743902,1072.743902,554.018293,258.115854,369.560976,280.213415,586.280488,...,1980.304878,"POLYGON ((11.28056 49.41777, 11.28065 49.41957...",9890,1.000000,0.758218,1.620569,1.137082,-0.310229,-0.692960,-0.487541
9891,665200,5476600,1.0,1107.400000,880.220000,445.480000,255.520000,354.760000,254.240000,533.640000,...,1749.300000,"POLYGON ((11.28065 49.41957, 11.28073 49.42137...",9891,1.000000,0.748374,1.590076,1.122284,-0.334870,-0.665526,-0.425481
9892,665400,5476200,1.0,270.941176,1056.941176,535.235294,253.529412,376.558824,266.882353,612.235294,...,2018.382353,"POLYGON ((11.28324 49.41592, 11.28332 49.41772...",9892,1.000000,0.785030,1.705279,1.177308,-0.354151,-0.709518,-0.474630


### Create a Parquet Version of the CSV file to save on space and increase load times for the dashboard

In [18]:
from pathlib import Path
import pyarrow.csv as pacsv
import pyarrow.parquet as pq

csv_path = Path("data_3x3/delta_table_2021_3x3.csv")
parquet_path = csv_path.with_suffix(".parquet")

table = pacsv.read_csv(csv_path)
pq.write_table(table, parquet_path)

print(f"Saved: {parquet_path}")
print(f"Rows: {table.num_rows:,} | Columns: {table.num_columns}")

Saved: data_3x3/delta_table_2021_3x3.parquet
Rows: 2,744,178 | Columns: 96
